In [ ]:
import json
import joblib
import datetime
import numpy as np
import pandas as pd
import os
import io
import mimetypes
import hashlib
import hmac
import math
import random
import uuid
from http.server import BaseHTTPRequestHandler, HTTPServer
from tensorflow.keras.models import load_model
from openai import OpenAI


from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm, cm
from reportlab.lib.colors import HexColor, Color
from reportlab.pdfgen.canvas import Canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

In [ ]:
FONT_PATH = 'assets/Arial Unicode.ttf'
pdfmetrics.registerFont(TTFont('ArialUni', FONT_PATH))
FONT_NAME = 'ArialUni'


SECRET_KEY = b'AnKa_S3ismIc_2025_!@#SealKey_XzQ9'
REPORTS_DB_FILE = 'pdf_reports.json'


LAST_BUILDING_DATA = {}
LAST_COLLAPSE_PROB = None


print("Loading model and preprocessors... Please wait 💅")
model = load_model('assets/model.keras')
model_columns = joblib.load('assets/model_columns.pkl')
scaler = joblib.load('assets/scaler.pkl')

llm_client = OpenAI(
    api_key="sk-aGkzeUjyrzjR1iO0SnMDpEHe5q6cyAHWPjgBlNlPvkAtD82s",
    base_url="https://api.gapgpt.app/v1"
)

stt_client = OpenAI(
    base_url="https://api.gapgpt.app/v1",
    api_key="sk-aGkzeUjyrzjR1iO0SnMDpEHe5q6cyAHWPjgBlNlPvkAtD82s"
)

ANKA_SYSTEM_PROMPT = """You are Anka, an expert AI assistant specialized EXCLUSIVELY in building safety, structural engineering, earthquake preparedness, architecture, and construction.

STRICT RULES:
1. You MUST ONLY answer questions related to:
   - buildings and construction,
   - structural engineering,
   - earthquake safety and earthquake preparedness,
   - architecture,
   - building materials,
   - building codes,
   - seismic design and retrofitting,
   - foundation types and soil mechanics,
   - building inspection,
   - emergency preparedness for earthquakes,
   - general earthquake science (magnitude scales, seismic waves, fault types, historical earthquakes) **as long as you connect them to building safety, damage patterns, or engineering lessons learned**,
   - and closely related civil engineering topics.
2. If a user asks ANYTHING outside these topics (politics, entertainment, coding, math, personal advice, etc.), you MUST politely decline and redirect them to building safety topics. Say something like: "I'm specialized in building safety and earthquake preparedness. I'd be happy to help you with questions about your building's structural integrity, earthquake readiness, or construction best practices!"
3. If the user asks about a specific earthquake event (for example: "What was the biggest earthquake in Turkey?" or "What happened in the 1999 İzmit earthquake?"), you SHOULD answer it briefly and clearly, focusing on:
   - basic facts (magnitude, location, date),
   - main damage patterns,
   - what happened to buildings and infrastructure,
   - and the key lessons for modern building safety and seismic design.
4. You have access to the user's building data (provided in the system context). Use this information proactively to give personalized, specific advice.
5. Be warm, professional, and encouraging. Use a natural conversational tone.
6. When giving safety recommendations, be specific and actionable.
7. If the collapse probability is high (>50%), be honest but not alarmist. Focus on actionable steps the user can take.
8. Support conversations in English, Farsi (Persian), and Turkish. Detect the user's language from their message and respond in the same language.
9. Keep responses concise but thorough. Use bullet points for lists.
10. If the user asks about their specific building, reference the actual data you have (floors, age, area, etc.) in your response.
11. You may discuss general earthquake science (magnitude scales, seismic waves, plate tectonics, historical events, etc.) as long as you relate it to building safety, structural behavior, risk, or preparedness.

BUILDING DATA CONTEXT (provided by the system — this is the user's actual building):
{building_context}

Based on the analysis, the estimated collapse probability for this building is {collapse_prob}%.

Use this data to provide personalized advice. Reference specific values when relevant."""



def add_features(df):
    df['seismic_stress'] = df['earthquake_magnitude'] * df['floors']
    df['building_mass_proxy'] = df['area'] * df['floors']
    df['age_height_ratio'] = df['building_age'] / (df['floors'] + 1)
    df['structural_risk_index'] = df['earthquake_magnitude'] * df['floors'] * (df['building_age'] + 1)
    df['seismic_energy'] = df['earthquake_magnitude'] ** 2 * df['floors']
    df['liquefaction_seismic'] = df['soil_liquefaction_risk'] * df['earthquake_magnitude']
    df['shear_wall_risk'] = (1 - df['Shear_wall']) * df['earthquake_magnitude']
    df['floors_area_density'] = df['floors'] / (df['area'] + 1)
    return df

def prepare_input(input_dict):
    df = pd.DataFrame([input_dict])
    liquefaction_map = {'low': 0, 'medium': 1, 'high': 2}
    df['soil_liquefaction_risk'] = df['soil_liquefaction_risk'].map(liquefaction_map)
    shear_wall_map = {'no': 0, 'yes': 1}
    df['Shear_wall'] = df['Shear_wall'].map(shear_wall_map)
    df = pd.get_dummies(df, columns=['structure_type', 'soil_type'])
    df = add_features(df)
    df = df.reindex(columns=model_columns, fill_value=0)
    num_cols = ['floors', 'building_age', 'area', 'earthquake_magnitude',
                'seismic_stress', 'building_mass_proxy', 'age_height_ratio',
                'structural_risk_index', 'seismic_energy', 'liquefaction_seismic',
                'shear_wall_risk', 'floors_area_density']
    df[num_cols] = scaler.transform(df[num_cols])
    return df.values.astype(np.float32)



BG_DARK      = HexColor('#0a0e1a')
BG_CARD      = HexColor('#111827')
BG_CARD_ALT  = HexColor('#1a2235')
ACCENT_CYAN  = HexColor('#00d4ff')
ACCENT_GOLD  = HexColor('#f5a623')
TEXT_WHITE    = HexColor('#f0f0f0')
TEXT_GRAY     = HexColor('#9ca3af')
TEXT_DIM      = HexColor('#6b7280')
BORDER_LINE  = HexColor('#1e293b')
GREEN_SAFE   = HexColor('#00ff88')
YELLOW_MOD   = HexColor('#ffea00')
ORANGE_HIGH  = HexColor('#ff9500')
RED_CRIT     = HexColor('#ff3333')



def _seal_hash(report_id, building_str, timestamp_str):
    """Generate HMAC-SHA256 for seal authenticity."""
    payload = f"{report_id}|{building_str}|{timestamp_str}".encode('utf-8')
    return hmac.new(SECRET_KEY, payload, hashlib.sha256).hexdigest()


def _ensure_reports_db():
    if not os.path.exists(REPORTS_DB_FILE):
        with open(REPORTS_DB_FILE, 'w', encoding='utf-8') as f:
            json.dump([], f, ensure_ascii=False, indent=2)

def _load_reports_db():
    _ensure_reports_db()
    try:
        with open(REPORTS_DB_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                return data
            return []
    except Exception:
        return []

def _save_reports_db(data):
    with open(REPORTS_DB_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def _store_pdf_report_record(record):
    reports = _load_reports_db()
    reports.append(record)
    _save_reports_db(reports)

def _find_pdf_report_record(year, month, day, hour, minute, second, report_id):
    reports = _load_reports_db()
    for record in reports:
        ts = record.get('created_at_parts', {})
        if (
            str(ts.get('year')) == str(year) and
            str(ts.get('month')) == str(month) and
            str(ts.get('day')) == str(day) and
            str(ts.get('hour')) == str(hour) and
            str(ts.get('minute')) == str(minute) and
            str(ts.get('second')) == str(second) and
            str(record.get('report_id', '')).upper() == str(report_id).upper()
        ):
            return record
    return None


def _draw_hologram(c, cx, cy, radius, report_id, seal_hash, timestamp_str):
    """
    Draw an advanced anti-forgery hologram seal on the canvas.
    cx, cy = center of seal
    radius = outer radius
    """
    c.saveState()

    for i in range(8):
        alpha = 0.03 - i * 0.003
        if alpha < 0:
            break
        r = radius + i * 2
        c.setStrokeColor(Color(0, 0.83, 1, alpha))
        c.setLineWidth(1.5)
        c.circle(cx, cy, r, stroke=1, fill=0)

    c.setFillColor(Color(0.04, 0.055, 0.1, 0.95))
    c.circle(cx, cy, radius, stroke=0, fill=1)

    random.seed(42)
    c.setFillColor(Color(0, 0.83, 1, 0.06))
    for _ in range(300):
        angle = random.uniform(0, 2 * math.pi)
        dist = random.uniform(0, radius * 0.92)
        dx = cx + dist * math.cos(angle)
        dy = cy + dist * math.sin(angle)
        c.circle(dx, dy, 0.4, stroke=0, fill=1)

    ring_colors = [
        (0.85, Color(0, 0.83, 1, 0.12)),
        (0.72, Color(0, 0.83, 1, 0.18)),
        (0.58, Color(0, 0.83, 1, 0.10)),
        (0.44, Color(0.96, 0.65, 0.14, 0.15)),
        (0.30, Color(0, 0.83, 1, 0.20)),
    ]
    for ratio, color in ring_colors:
        c.setStrokeColor(color)
        c.setLineWidth(0.6)
        c.circle(cx, cy, radius * ratio, stroke=1, fill=0)

    num_rays = 36
    c.setStrokeColor(Color(0, 0.83, 1, 0.07))
    c.setLineWidth(0.3)
    for i in range(num_rays):
        angle = (2 * math.pi / num_rays) * i
        x1 = cx + radius * 0.25 * math.cos(angle)
        y1 = cy + radius * 0.25 * math.sin(angle)
        x2 = cx + radius * 0.90 * math.cos(angle)
        y2 = cy + radius * 0.90 * math.sin(angle)
        c.line(x1, y1, x2, y2)

    outer_text = f"  ANKA SEISMIC REPORT  •  VERIFIED  •  {timestamp_str}  •  "
    c.setFillColor(Color(0, 0.83, 1, 0.55))
    c.setFont(FONT_NAME, 5)
    text_radius = radius * 0.82
    char_angle_step = 360 / max(len(outer_text), 1)
    for i, ch in enumerate(outer_text):
        angle_deg = i * char_angle_step - 90
        angle_rad = math.radians(angle_deg)
        tx = cx + text_radius * math.cos(angle_rad)
        ty = cy + text_radius * math.sin(angle_rad)
        c.saveState()
        c.translate(tx, ty)
        c.rotate(angle_deg + 90)
        c.drawCentredString(0, 0, ch)
        c.restoreState()

    inner_text = f"  ID:{report_id[:12]}  •  SHA256:{seal_hash[:16]}  •  "
    c.setFillColor(Color(0.96, 0.65, 0.14, 0.45))
    c.setFont(FONT_NAME, 3.8)
    inner_radius = radius * 0.58
    char_angle_step2 = 360 / max(len(inner_text), 1)
    for i, ch in enumerate(inner_text):
        angle_deg = -i * char_angle_step2 + 90
        angle_rad = math.radians(angle_deg)
        tx = cx + inner_radius * math.cos(angle_rad)
        ty = cy + inner_radius * math.sin(angle_rad)
        c.saveState()
        c.translate(tx, ty)
        c.rotate(angle_deg - 90)
        c.drawCentredString(0, 0, ch)
        c.restoreState()

    c.setStrokeColor(Color(0, 0.83, 1, 0.15))
    c.setLineWidth(0.35)
    petals = 12
    for i in range(petals):
        angle = (2 * math.pi / petals) * i
        r_inner = radius * 0.18
        r_outer = radius * 0.42
        a1 = angle - math.pi / petals * 0.4
        a2 = angle + math.pi / petals * 0.4
        p = c.beginPath()
        p.moveTo(cx, cy)
        cp1x = cx + r_outer * math.cos(a1)
        cp1y = cy + r_outer * math.sin(a1)
        cp2x = cx + r_outer * math.cos(a2)
        cp2y = cy + r_outer * math.sin(a2)
        endx = cx + r_inner * math.cos(angle + math.pi / petals)
        endy = cy + r_inner * math.sin(angle + math.pi / petals)
        p.curveTo(cp1x, cp1y, cp2x, cp2y, cx, cy)
        c.drawPath(p, stroke=1, fill=0)

    c.setStrokeColor(Color(0, 0.83, 1, 0.08))
    c.setLineWidth(0.25)
    R_sp = radius * 0.38
    r_sp = radius * 0.14
    d_sp = radius * 0.32
    p = c.beginPath()
    first = True
    for t in range(0, 628, 1):
        tt = t / 100.0
        x = cx + (R_sp - r_sp) * math.cos(tt) + d_sp * math.cos((R_sp - r_sp) / r_sp * tt)
        y = cy + (R_sp - r_sp) * math.sin(tt) - d_sp * math.sin((R_sp - r_sp) / r_sp * tt)
        if first:
            p.moveTo(x, y)
            first = False
        else:
            p.lineTo(x, y)
    c.drawPath(p, stroke=1, fill=0)

    c.setFillColor(Color(0.04, 0.06, 0.12, 1))
    c.circle(cx, cy, radius * 0.18, stroke=0, fill=1)
    c.setStrokeColor(Color(0, 0.83, 1, 0.5))
    c.setLineWidth(1)
    c.circle(cx, cy, radius * 0.18, stroke=1, fill=0)

    c.setFillColor(ACCENT_CYAN)
    c.setFont(FONT_NAME, 6.5)
    c.drawCentredString(cx, cy + 2, "VERIFIED")

    c.setFillColor(ACCENT_GOLD)
    c.setFont(FONT_NAME, 5)
    c.drawCentredString(cx, cy - 6, "ANKA")

    c.setFillColor(Color(0.6, 0.65, 0.7, 0.4))
    c.setFont(FONT_NAME, 3)
    c.drawCentredString(cx, cy - radius - 10, f"SHA256: {seal_hash[:32]}...")
    c.drawCentredString(cx, cy - radius - 16, f"Report: {report_id}")

    c.restoreState()


def generate_pdf_report(input_data, prob, recommendations, lang='en'):
    """Generate a professional dark-themed PDF report with hologram seal."""

    def fix_rtl(text):
        """Basic RTL fix for Persian text without external bidi libs."""
        lines = text.split('\n')
        fixed = []
        for line in lines:
            if any('\u0600' <= ch <= '\u06FF' for ch in line):
                words = line.split(' ')
                fixed.append(' '.join(reversed(words)))
            else:
                fixed.append(line)
        return '\n'.join(fixed)

    now = datetime.datetime.now()
    time_str = now.strftime("%H_%M_%S")
    timestamp_str = now.strftime("%Y-%m-%d %H:%M:%S")
    filename = f"report_{time_str}.pdf"
    report_id = str(uuid.uuid4()).upper()[:16]

    building_str = json.dumps(input_data, sort_keys=True)
    seal_hash = _seal_hash(report_id, building_str, timestamp_str)

    W, H = A4
    c = Canvas(filename, pagesize=A4)

    c.setFillColor(BG_DARK)
    c.rect(0, 0, W, H, stroke=0, fill=1)

    c.setStrokeColor(ACCENT_CYAN)
    c.setLineWidth(3)
    c.line(0, H - 8, W, H - 8)

    c.setStrokeColor(ACCENT_GOLD)
    c.setLineWidth(0.8)
    c.line(0, H - 12, W, H - 12)

    y = H - 40

    logo_path = 'logo.png'
    if os.path.exists(logo_path):
        try:
            c.drawImage(logo_path, 30, y - 35, width=45, height=45,
                        preserveAspectRatio=True, mask='auto')
        except:
            c.setFillColor(ACCENT_CYAN)
            c.setFont(FONT_NAME, 28)
            c.drawString(35, y - 25, "A")
    else:
        c.setFillColor(ACCENT_CYAN)
        c.setFont(FONT_NAME, 28)
        c.drawString(35, y - 25, "A")

    c.setFillColor(TEXT_WHITE)
    c.setFont(FONT_NAME, 20)
    c.drawString(85, y - 8, "Anka Seismic Report")

    c.setFillColor(TEXT_GRAY)
    c.setFont(FONT_NAME, 8)
    c.drawString(85, y - 22, f"Earthquake Collapse Risk Assessment")

    c.setFillColor(ACCENT_CYAN)
    c.setFont(FONT_NAME, 7)
    c.drawRightString(W - 30, y - 8, f"Report ID: {report_id}")
    c.setFillColor(TEXT_DIM)
    c.setFont(FONT_NAME, 7)
    c.drawRightString(W - 30, y - 20, f"Generated: {timestamp_str}")

    y -= 55

    c.setStrokeColor(BORDER_LINE)
    c.setLineWidth(0.5)
    c.line(30, y, W - 30, y)
    y -= 25

    if prob <= 25:
        risk_color = GREEN_SAFE
        risk_label = "LOW RISK"
        risk_icon = "🟢"
    elif prob <= 50:
        risk_color = YELLOW_MOD
        risk_label = "MODERATE RISK"
        risk_icon = "🟡"
    elif prob <= 75:
        risk_color = ORANGE_HIGH
        risk_label = "HIGH RISK"
        risk_icon = "🟠"
    else:
        risk_color = RED_CRIT
        risk_label = "CRITICAL RISK"
        risk_icon = "🔴"

    card_x = 30
    card_w = W - 60
    card_h = 70
    c.setFillColor(BG_CARD)
    c.roundRect(card_x, y - card_h, card_w, card_h, 8, stroke=0, fill=1)

    c.setFillColor(risk_color)
    c.rect(card_x, y - card_h, 4, card_h, stroke=0, fill=1)

    c.setFillColor(risk_color)
    c.setFont(FONT_NAME, 36)
    c.drawString(card_x + 20, y - 48, f"{prob:.1f}%")

    c.setFont(FONT_NAME, 14)
    c.drawString(card_x + 140, y - 35, risk_label)

    c.setFillColor(TEXT_GRAY)
    c.setFont(FONT_NAME, 9)
    c.drawString(card_x + 140, y - 52, "Collapse Probability")

    c.setFont(FONT_NAME, 28)
    c.drawRightString(W - 50, y - 45, risk_icon)

    y -= card_h + 20

    c.setFillColor(ACCENT_CYAN)
    c.setFont(FONT_NAME, 12)
    c.drawString(30, y, "Building Specifications")
    y -= 5

    c.setStrokeColor(ACCENT_CYAN)
    c.setLineWidth(0.5)
    c.line(30, y, 200, y)
    y -= 15

    label_map = {
        'floors': 'Floors',
        'building_age': 'Building Age (yrs)',
        'area': 'Area (m²)',
        'earthquake_magnitude': 'Earthquake Magnitude',
        'structure_type': 'Structure Type',
        'soil_type': 'Soil Type',
        'soil_liquefaction_risk': 'Liquefaction Risk',
        'Shear_wall': 'Shear Wall',
    }

    row_h = 20
    col1_x = 40
    col2_x = 220
    items = list(input_data.items())

    for i, (key, val) in enumerate(items):
        row_y = y - i * row_h
        if i % 2 == 0:
            c.setFillColor(BG_CARD)
        else:
            c.setFillColor(BG_CARD_ALT)
        c.rect(30, row_y - 5, W - 60, row_h, stroke=0, fill=1)

        c.setFillColor(TEXT_GRAY)
        c.setFont(FONT_NAME, 9)
        display_key = label_map.get(key, key)
        c.drawString(col1_x, row_y + 3, display_key)

        c.setFillColor(TEXT_WHITE)
        c.setFont(FONT_NAME, 9)
        c.drawString(col2_x, row_y + 3, str(val))

    y -= len(items) * row_h + 20

    c.setFillColor(ACCENT_GOLD)
    c.setFont(FONT_NAME, 12)
    c.drawString(30, y, "Safety Recommendations")
    y -= 5
    c.setStrokeColor(ACCENT_GOLD)
    c.setLineWidth(0.5)
    c.line(30, y, 220, y)
    y -= 15

    c.setFont(FONT_NAME, 8)
    for rec in recommendations:
        display_rec = fix_rtl(rec) if lang == 'fa' else rec

        max_chars = 90
        lines = []
        while len(display_rec) > max_chars:
            split_at = display_rec.rfind(' ', 0, max_chars)
            if split_at == -1:
                split_at = max_chars
            lines.append(display_rec[:split_at])
            display_rec = display_rec[split_at:].strip()
        lines.append(display_rec)

        for j, line in enumerate(lines):
            if y < 160:
                c.showPage()
                c.setFillColor(BG_DARK)
                c.rect(0, 0, W, H, stroke=0, fill=1)
                y = H - 40

            if j == 0:
                c.setFillColor(ACCENT_GOLD)
                c.drawString(35, y, "⚠")
                c.setFillColor(TEXT_WHITE)
                c.drawString(50, y, line)
            else:
                c.setFillColor(TEXT_WHITE)
                c.drawString(50, y, line)
            y -= 13

        y -= 5

    seal_radius = 55
    seal_cx = W / 2
    seal_cy = max(y - seal_radius - 15, seal_radius + 40)

    if seal_cy - seal_radius < 50:
        c.showPage()
        c.setFillColor(BG_DARK)
        c.rect(0, 0, W, H, stroke=0, fill=1)
        seal_cy = H / 2

    _draw_hologram(c, seal_cx, seal_cy, seal_radius, report_id, seal_hash, timestamp_str)

    footer_y = 25

    c.setStrokeColor(BORDER_LINE)
    c.setLineWidth(0.3)
    c.line(30, footer_y + 12, W - 30, footer_y + 12)

    c.setFillColor(TEXT_DIM)
    c.setFont(FONT_NAME, 6)
    c.drawCentredString(W / 2, footer_y,
                        f"© {now.year} Anka Seismic Intelligence  •  This report is digitally sealed and tamper-evident  •  Verify: {seal_hash[:24]}")

    c.setStrokeColor(ACCENT_CYAN)
    c.setLineWidth(2)
    c.line(0, 5, W, 5)

    c.save()

    report_record = {
        "report_id": report_id,
        "filename": filename,
        "created_at": timestamp_str,
        "created_at_parts": {
            "year": now.year,
            "month": now.month,
            "day": now.day,
            "hour": now.hour,
            "minute": now.minute,
            "second": now.second
        },
        "building_data": input_data,
        "collapse_probability": round(float(prob), 2),
        "seal_hash": seal_hash
    }
    _store_pdf_report_record(report_record)

    return filename, report_id, timestamp_str


class RequestHandler(BaseHTTPRequestHandler):
    def do_OPTIONS(self):
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.end_headers()

    def do_GET(self):
        STATIC_FILES = {
            'simulation.html',
            'style.css',
            'AI.html',
            'pdf.html'
        }

        MIME_MAP = {
            '.html': 'text/html; charset=utf-8',
            '.css':  'text/css; charset=utf-8',
            '.js':   'application/javascript; charset=utf-8',
            '.pdf':  'application/pdf',
            '.png':  'image/png',
            '.jpg':  'image/jpeg',
            '.svg':  'image/svg+xml',
            '.json': 'application/json; charset=utf-8',
        }

        path = self.path.strip('/')

        if path == '' or path == 'index.html':
            path = 'index.html'

        if path.startswith('download/'):
            filename = path.split('/', 1)[1]
            if os.path.exists(filename):
                mime_type = mimetypes.guess_type(filename)[0] or 'application/octet-stream'
                self.send_response(200)
                self.send_header('Content-Type', mime_type)
                self.send_header('Content-Disposition', f'attachment; filename="{filename}"')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                with open(filename, 'rb') as f:
                    self.wfile.write(f.read())
            else:
                self.send_response(404)
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(b'File not found')
            return

        if path in STATIC_FILES or path == 'index.html':
            if os.path.exists(path):
                ext = os.path.splitext(path)[1]
                content_type = MIME_MAP.get(ext, 'text/plain; charset=utf-8')
                self.send_response(200)
                self.send_header('Content-Type', content_type)
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                with open(path, 'rb') as f:
                    self.wfile.write(f.read())
            else:
                self.send_response(404)
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(b'File not found')
            return

        self.send_response(404)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.end_headers()
        self.wfile.write(b'Not found')

    def do_POST(self):
        content_length = int(self.headers.get('Content-Length', 0))
        body = self.rfile.read(content_length)

        if self.path == '/predict':
            try:
                data = json.loads(body)
                future_years = int(data.get('future_years', 0))
                lang = data.get('lang', 'en')
                building_age = int(data['building_age']) + future_years

                input_data = {
                    'floors': int(data['floors']),
                    'building_age': building_age,
                    'area': float(data['area']),
                    'earthquake_magnitude': float(data['earthquake_magnitude']),
                    'structure_type': data['structure_type'],
                    'soil_type': data['soil_type'],
                    'soil_liquefaction_risk': data['soil_liquefaction_risk'],
                    'Shear_wall': data['Shear_wall'],
                }

                X = prepare_input(input_data)
                prob = float(model.predict(X)[0][0]) * 100


                global LAST_BUILDING_DATA, LAST_COLLAPSE_PROB
                LAST_BUILDING_DATA = input_data
                LAST_COLLAPSE_PROB = prob

                if prob <= 25:
                    risk_color = "#00ff88"
                elif prob <= 50:
                    risk_color = "#ffea00"
                elif prob <= 75:
                    risk_color = "#ff9500"
                else:
                    risk_color = "#ff3333"


                rec_dict = {

                    "critical_collapse": {
                        "en": "🔴 CRITICAL: Model predicts imminent structural collapse risk. Immediate evacuation is mandatory. Do not re-enter until a licensed structural engineer clears the building.",
                        "fa": "🔴 بحرانی: مدل احتمال فروریزش سازه را بسیار بالا پیش‌بینی می‌کند. تخلیه فوری الزامی است. تا تأیید مهندس سازه مجاز، از ورود به ساختمان خودداری کنید.",
                        "tr": "🔴 KRİTİK: Model ani yapısal çökme riski öngörüyor. Derhal tahliye zorunludur. Lisanslı bir yapı mühendisi onaylamadan binaya girmeyin."
                    },
                    "high_prob_old": {
                        "en": "🔴 CRITICAL: High failure probability combined with aging structure (>30 yrs). Immediate seismic retrofit by a certified engineer is required.",
                        "fa": "🔴 بحرانی: احتمال خرابی بالا در ترکیب با سازه فرسوده (بیش از ۳۰ سال). بازسازی لرزه‌ای فوری توسط مهندس معتمد الزامی است.",
                        "tr": "🔴 KRİTİK: Yaşlı yapıda (>30 yıl) yüksek hasar olasılığı. Sertifikalı mühendis tarafından acil sismik güçlendirme gereklidir."
                    },
                    "high_prob": {
                        "en": "🟠 HIGH RISK: Model predicts high structural vulnerability. A full seismic evaluation by a certified structural engineer is strongly advised.",
                        "fa": "🟠 خطر بالا: مدل آسیب‌پذیری سازه‌ای بالایی پیش‌بینی می‌کند. ارزیابی لرزه‌ای کامل توسط مهندس سازه توصیه می‌شود.",
                        "tr": "🟠 YÜKSEK RİSK: Model yüksek yapısal kırılganlık öngörüyor. Sertifikalı bir yapı mühendisi tarafından tam sismik değerlendirme yapılması önerilir."
                    },


                    "liquefaction_high_tall": {
                        "en": "🔴 CRITICAL: High liquefaction risk beneath a tall multi-story building. Emergency geotechnical assessment and deep pile foundation inspection are required immediately.",
                        "fa": "🔴 بحرانی: خطر روانگرایی بالا زیر ساختمان بلند چندطبقه. ارزیابی ژئوتکنیک اضطراری و بازرسی شمع‌های عمیق فوراً لازم است.",
                        "tr": "🔴 KRİTİK: Yüksek katlı binada yüksek sıvılaşma riski. Acil geoteknik değerlendirme ve derin kazık temeli incelemesi gereklidir."
                    },
                    "liquefaction_high": {
                        "en": "🟠 HIGH RISK: High soil liquefaction risk detected. Foundation reinforcement and geotechnical soil stabilization are strongly advised.",
                        "fa": "🟠 خطر بالا: خطر روانگرایی خاک بالا شناسایی شد. تقویت پی و پایدارسازی ژئوتکنیک خاک اکیداً توصیه می‌شود.",
                        "tr": "🟠 YÜKSEK RİSK: Yüksek zemin sıvılaşma riski tespit edildi. Temel güçlendirme ve geoteknik zemin stabilizasyonu şiddetle tavsiye edilir."
                    },
                    "liquefaction_medium_tall": {
                        "en": "🟡 WARNING: Moderate liquefaction risk under a multi-story structure. Geotechnical assessment and foundation review are recommended.",
                        "fa": "🟡 هشدار: خطر متوسط روانگرایی زیر سازه چندطبقه. ارزیابی ژئوتکنیک و بررسی پی توصیه می‌شود.",
                        "tr": "🟡 UYARI: Çok katlı yapı altında orta düzey sıvılaşma riski. Geoteknik değerlendirme ve temel incelemesi önerilir."
                    },


                    "masonry_major_quake": {
                        "en": "🔴 CRITICAL: Unreinforced masonry under major seismic load (M≥6.5) is extremely dangerous. Immediate structural reinforcement or building relocation is advised.",
                        "fa": "🔴 بحرانی: بنایی غیرمسلح در برابر زلزله شدید (M≥۶.۵) بسیار خطرناک است. تقویت فوری سازه یا تخلیه ساختمان توصیه می‌شود.",
                        "tr": "🔴 KRİTİK: Büyük sismik yük altında (M≥6.5) güçlendirilmemiş yığma yapı son derece tehlikelidir. Acil yapısal güçlendirme veya tahliye önerilir."
                    },
                    "masonry_moderate_quake": {
                        "en": "🟠 HIGH RISK: Masonry structures are highly vulnerable to earthquakes (M≥5.5). Adding reinforced tie beams and shear walls is strongly recommended.",
                        "fa": "🟠 خطر بالا: سازه‌های بنایی در برابر زلزله (M≥۵.۵) بسیار آسیب‌پذیرند. افزودن تیرهای کلاف مسلح و دیوار برشی اکیداً توصیه می‌شود.",
                        "tr": "🟠 YÜKSEK RİSK: Yığma yapılar depremlere (M≥5.5) karşı oldukça savunmasızdır. Güçlendirilmiş bağ kirişleri ve perde duvarlar eklenmesi şiddetle tavsiye edilir."
                    },
                    "masonry_old": {
                        "en": "🟠 HIGH RISK: Old masonry building (>25 yrs) with no modern seismic detailing. Comprehensive structural assessment is urgently needed.",
                        "fa": "🟠 خطر بالا: ساختمان بنایی قدیمی (بیش از ۲۵ سال) بدون جزئیات لرزه‌ای مدرن. ارزیابی جامع سازه‌ای فوری لازم است.",
                        "tr": "🟠 YÜKSEK RİSK: Modern sismik detaylandırma olmayan eski yığma bina (>25 yıl). Kapsamlı yapısal değerlendirme acilen gereklidir."
                    },


                    "wood_high_quake": {
                        "en": "🟠 HIGH RISK: Wooden structures under strong seismic activity (M≥6.0) are prone to collapse. Ensure proper anchoring, bracing, and connection reinforcement.",
                        "fa": "🟠 خطر بالا: سازه‌های چوبی در برابر زلزله شدید (M≥۶.۰) مستعد فروریزش هستند. اتصالات، مهاربندی و لنگرگذاری مناسب را تأمین کنید.",
                        "tr": "🟠 YÜKSEK RİSK: Güçlü sismik aktivite altında (M≥6.0) ahşap yapılar çökmeye eğilimlidir. Uygun ankraj, bağlantı ve çapraz destek güçlendirmesi sağlayın."
                    },
                    "wood_soft_soil": {
                        "en": "🟡 WARNING: Wooden structure on soft/clay soil amplifies seismic response. Foundation anchoring and soil improvement are recommended.",
                        "fa": "🟡 هشدار: سازه چوبی روی خاک نرم/رسی پاسخ لرزه‌ای را تشدید می‌کند. لنگرگذاری پی و بهسازی خاک توصیه می‌شود.",
                        "tr": "🟡 UYARI: Yumuşak/killi zemin üzerindeki ahşap yapı sismik tepkiyi artırır. Temel ankrajı ve zemin iyileştirmesi önerilir."
                    },


                    "soft_soil_tall_heavy": {
                        "en": "🔴 CRITICAL: Tall heavy structure (>6 floors) on soft/clay soil. Soil amplification significantly increases collapse risk. Deep pile foundation verification is mandatory.",
                        "fa": "🔴 بحرانی: سازه بلند و سنگین (بیش از ۶ طبقه) روی خاک نرم/رسی. تقویت خاک احتمال فروریزش را به‌شدت افزایش می‌دهد. تأیید شمع‌های عمیق الزامی است.",
                        "tr": "🔴 KRİTİK: Yumuşak/killi zemin üzerinde yüksek ağır yapı (>6 kat). Zemin amplifikasyonu çökme riskini önemli ölçüde artırır. Derin kazık temeli doğrulaması zorunludur."
                    },
                    "soft_soil_heavy": {
                        "en": "🟠 HIGH RISK: Concrete/Steel structure on soft or clay soil. Verify adequate pile depth and foundation integrity to prevent differential settlement.",
                        "fa": "🟠 خطر بالا: سازه بتنی/فولادی روی خاک نرم یا رسی. عمق کافی شمع و یکپارچگی پی را برای جلوگیری از نشست تفاضلی تأیید کنید.",
                        "tr": "🟠 YÜKSEK RİSK: Yumuşak veya killi zemin üzerinde beton/çelik yapı. Diferansiyel oturmayı önlemek için yeterli kazık derinliği ve temel bütünlüğünü doğrulayın."
                    },
                    "soft_soil_quake": {
                        "en": "🟡 WARNING: Soft/clay soil significantly amplifies seismic waves. Even moderate earthquakes can cause disproportionate structural damage.",
                        "fa": "🟡 هشدار: خاک نرم/رسی امواج لرزه‌ای را به‌طور قابل‌توجهی تقویت می‌کند. حتی زلزله‌های متوسط می‌توانند آسیب سازه‌ای نامتناسبی ایجاد کنند.",
                        "tr": "🟡 UYARI: Yumuşak/killi zemin sismik dalgaları önemli ölçüde yükseltir. Orta şiddetteki depremler bile orantısız yapısal hasara yol açabilir."
                    },


                    "no_shear_wall_critical": {
                        "en": "🔴 CRITICAL: Tall building (>7 floors) with no shear walls is critically vulnerable to lateral seismic forces. Immediate structural bracing or shear wall retrofit is required.",
                        "fa": "🔴 بحرانی: ساختمان بلند (بیش از ۷ طبقه) بدون دیوار برشی در برابر نیروهای جانبی لرزه‌ای بسیار آسیب‌پذیر است. مهاربندی فوری یا نصب دیوار برشی الزامی است.",
                        "tr": "🔴 KRİTİK: Perde duvarı olmayan yüksek bina (>7 kat) yanal sismik kuvvetlere karşı kritik derecede savunmasızdır. Acil yapısal bağlantı veya perde duvar güçlendirmesi gereklidir."
                    },
                    "no_shear_wall_high": {
                        "en": "🟠 HIGH RISK: Building with no shear walls (>4 floors) has insufficient lateral resistance. Retrofitting with bracing or shear walls is strongly recommended.",
                        "fa": "🟠 خطر بالا: ساختمان بدون دیوار برشی (بیش از ۴ طبقه) مقاومت جانبی کافی ندارد. مقاوم‌سازی با مهاربند یا دیوار برشی اکیداً توصیه می‌شود.",
                        "tr": "🟠 YÜKSEK RİSK: Perde duvarı olmayan bina (>4 kat) yetersiz yanal dirence sahip. Bağlantı veya perde duvarla güçlendirme şiddetle tavsiye edilir."
                    },
                    "no_shear_wall": {
                        "en": "🟡 WARNING: Absence of shear walls increases lateral drift under seismic loads. Consider adding bracing or shear walls as a precautionary measure.",
                        "fa": "🟡 هشدار: نبود دیوار برشی انحراف جانبی در بارهای لرزه‌ای را افزایش می‌دهد. افزودن مهاربند یا دیوار برشی به‌عنوان اقدام پیشگیرانه توصیه می‌شود.",
                        "tr": "🟡 UYARI: Perde duvarı yokluğu sismik yükler altında yanal ötelenmeyi artırır. Önlem olarak bağlantı veya perde duvar eklenmesi düşünülmelidir."
                    },
                    "no_shear_wall_old": {
                        "en": "🟠 HIGH RISK: Old building (>25 yrs) with no shear walls — compounded vulnerability. Structural upgrade before the next seismic event is strongly advised.",
                        "fa": "🟠 خطر بالا: ساختمان قدیمی (بیش از ۲۵ سال) بدون دیوار برشی — آسیب‌پذیری مضاعف. ارتقای سازه‌ای پیش از رویداد لرزه‌ای بعدی اکیداً توصیه می‌شود.",
                        "tr": "🟠 YÜKSEK RİSK: Perde duvarı olmayan eski bina (>25 yıl) — bileşik kırılganlık. Bir sonraki sismik olaydan önce yapısal yükseltme şiddetle tavsiye edilir."
                    },


                    "very_old_building": {
                        "en": "🟠 HIGH RISK: Building age exceeds 50 years. Pre-modern seismic codes likely apply. Comprehensive structural audit and possible full retrofit are strongly recommended.",
                        "fa": "🟠 خطر بالا: عمر ساختمان بیش از ۵۰ سال است. احتمالاً آیین‌نامه‌های لرزه‌ای قدیمی اعمال شده. ممیزی جامع سازه‌ای و احتمالاً بازسازی کامل اکیداً توصیه می‌شود.",
                        "tr": "🟠 YÜKSEK RİSK: Bina yaşı 50 yılı aşıyor. Muhtemelen modern öncesi sismik kodlar uygulanmış. Kapsamlı yapısal denetim ve olası tam güçlendirme şiddetle tavsiye edilir."
                    },
                    "old_building": {
                        "en": "🟡 WARNING: Building is over 30 years old. Seismic standards have evolved significantly. A structural inspection to verify compliance with current codes is advised.",
                        "fa": "🟡 هشدار: ساختمان بیش از ۳۰ سال قدمت دارد. استانداردهای لرزه‌ای به‌طور قابل‌توجهی تکامل یافته‌اند. بازرسی سازه‌ای برای تأیید انطباق با آیین‌نامه‌های جاری توصیه می‌شود.",
                        "tr": "🟡 UYARI: Bina 30 yıldan eski. Sismik standartlar önemli ölçüde gelişti. Mevcut kodlara uygunluğu doğrulamak için yapısal inceleme tavsiye edilir."
                    },


                    "high_floors_soft_soil_quake": {
                        "en": "🔴 CRITICAL: Tall building on soft soil under significant seismic load — triple compounding risk. Dynamic soil-structure interaction analysis by a specialist is mandatory.",
                        "fa": "🔴 بحرانی: ساختمان بلند روی خاک نرم در معرض بار لرزه‌ای قابل‌توجه — خطر سه‌گانه مضاعف. تحلیل دینامیکی اندرکنش خاک-سازه توسط متخصص الزامی است.",
                        "tr": "🔴 KRİTİK: Önemli sismik yük altında yumuşak zemin üzerinde yüksek bina — üçlü bileşik risk. Uzman tarafından dinamik zemin-yapı etkileşim analizi zorunludur."
                    },
                    "large_area_soft_soil": {
                        "en": "🟡 WARNING: Large floor area on soft/clay soil increases differential settlement risk. Uniform foundation design and soil improvement are recommended.",
                        "fa": "🟡 هشدار: مساحت زیاد روی خاک نرم/رسی خطر نشست تفاضلی را افزایش می‌دهد. طراحی یکنواخت پی و بهسازی خاک توصیه می‌شود.",
                        "tr": "🟡 UYARI: Yumuşak/killi zemin üzerinde büyük taban alanı diferansiyel oturma riskini artırır. Tekdüze temel tasarımı ve zemin iyileştirmesi önerilir."
                    },
                    "future_age_risk": {
                        "en": "🟡 WARNING: Projected building age significantly increases long-term structural risk. Plan for preventive maintenance and periodic seismic re-evaluation.",
                        "fa": "🟡 هشدار: عمر پیش‌بینی‌شده ساختمان خطر سازه‌ای بلندمدت را به‌طور قابل‌توجهی افزایش می‌دهد. برنامه‌ریزی برای نگهداری پیشگیرانه و ارزیابی مجدد لرزه‌ای دوره‌ای توصیه می‌شود.",
                        "tr": "🟡 UYARI: Öngörülen bina yaşı uzun vadeli yapısal riski önemli ölçüde artırıyor. Önleyici bakım ve periyodik sismik yeniden değerlendirme planlanması önerilir."
                    },


                    "safe": {
                        "en": "✅ RELATIVELY SAFE: Building appears structurally adequate under the given conditions. Regular maintenance, periodic inspections, and adherence to local seismic codes are still advised.",
                        "fa": "✅ نسبتاً ایمن: ساختمان در شرایط داده‌شده از نظر سازه‌ای مناسب به نظر می‌رسد. نگهداری منظم، بازرسی‌های دوره‌ای و رعایت آیین‌نامه‌های لرزه‌ای محلی همچنان توصیه می‌شود.",
                        "tr": "✅ GÖRECE GÜVENLİ: Bina, verilen koşullar altında yapısal olarak yeterli görünüyor. Düzenli bakım, periyodik denetimler ve yerel sismik kodlara uyum hâlâ tavsiye edilir."
                    }
                }


                recs_keys = []

                prob_val        = prob
                floors_val      = input_data['floors']
                age_val         = building_age
                mag_val         = input_data['earthquake_magnitude']
                soil_val        = input_data['soil_type']
                struct_val      = input_data['structure_type']
                shear_val       = input_data['Shear_wall']
                liq_val         = input_data['soil_liquefaction_risk']
                area_val        = float(input_data['area'])
                future_val      = future_years

                soft_soil       = soil_val in ['soft_soil', 'clay']
                heavy_struct    = struct_val in ['concrete', 'steel']
                is_masonry      = struct_val == 'masnory'
                is_wood         = struct_val == 'wood'
                no_shear        = shear_val == 'no'

                if prob_val > 85:
                    recs_keys.append("critical_collapse")
                elif prob_val > 65 and age_val > 30:
                    recs_keys.append("high_prob_old")
                elif prob_val > 70:
                    recs_keys.append("high_prob")

                if liq_val == 'high' and floors_val > 5:
                    recs_keys.append("liquefaction_high_tall")
                elif liq_val == 'high':
                    recs_keys.append("liquefaction_high")
                elif liq_val == 'medium' and floors_val > 3:
                    recs_keys.append("liquefaction_medium_tall")

                if is_masonry:
                    if mag_val >= 6.5:
                        recs_keys.append("masonry_major_quake")
                    elif mag_val >= 5.5:
                        recs_keys.append("masonry_moderate_quake")
                    if age_val > 25:
                        recs_keys.append("masonry_old")

                if is_wood:
                    if mag_val >= 6.0:
                        recs_keys.append("wood_high_quake")
                    if soft_soil:
                        recs_keys.append("wood_soft_soil")

                if soft_soil:
                    if heavy_struct and floors_val > 6:
                        recs_keys.append("soft_soil_tall_heavy")
                    elif heavy_struct and floors_val > 3:
                        recs_keys.append("soft_soil_heavy")
                    elif mag_val >= 5.0:
                        recs_keys.append("soft_soil_quake")
                    if area_val > 500:
                        recs_keys.append("large_area_soft_soil")

                if no_shear:
                    if floors_val > 7:
                        recs_keys.append("no_shear_wall_critical")
                    elif floors_val > 4:
                        recs_keys.append("no_shear_wall_high")
                    elif floors_val > 2:
                        recs_keys.append("no_shear_wall")
                    if age_val > 25:
                        recs_keys.append("no_shear_wall_old")

                if soft_soil and floors_val > 7 and mag_val >= 5.5:
                    recs_keys.append("high_floors_soft_soil_quake")

                if age_val > 50:
                    recs_keys.append("very_old_building")
                elif age_val > 30 and "high_prob_old" not in recs_keys:
                    recs_keys.append("old_building")

                if future_val > 10 and age_val > 40:
                    recs_keys.append("future_age_risk")


                if len(recs_keys) == 0:
                    recs_keys.append("safe")

                seen = set()
                recs_keys = [r for r in recs_keys if not (r in seen or seen.add(r))]

                recommendations = [rec_dict[r].get(lang, rec_dict[r]["en"]) for r in recs_keys]

                pdf_file, report_id, created_at = generate_pdf_report(input_data, prob, recommendations, lang=lang)

                response = {
                    'probability': round(prob, 2),
                    'color': risk_color,
                    'recommendations': recommendations,
                    'pdf_file': pdf_file,
                    'report_id': report_id,
                    'created_at': created_at,
                }

                self.send_response(200)
                self.send_header('Content-Type', 'application/json')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(json.dumps(response).encode())

            except Exception as e:
                self.send_response(500)
                self.send_header('Content-Type', 'application/json')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(json.dumps({'error': str(e)}).encode())
            return

        if self.path == '/verify-pdf':
            try:
                data = json.loads(body)

                year = data.get('year')
                month = data.get('month')
                day = data.get('day')
                hour = data.get('hour')
                minute = data.get('minute')
                second = data.get('second')
                report_id = data.get('report_id')

                required_fields = [year, month, day, hour, minute, second, report_id]
                if any(v is None or str(v).strip() == '' for v in required_fields):
                    self.send_response(400)
                    self.send_header('Content-Type', 'application/json')
                    self.send_header('Access-Control-Allow-Origin', '*')
                    self.end_headers()
                    self.wfile.write(json.dumps({
                        'valid': False,
                        'error': 'Missing required fields'
                    }).encode())
                    return

                record = _find_pdf_report_record(year, month, day, hour, minute, second, report_id)

                if record:
                    response = {
                        'valid': True,
                        'report': {
                            'report_id': record.get('report_id'),
                            'created_at': record.get('created_at'),
                            'building_data': record.get('building_data', {}),
                            'collapse_probability': record.get('collapse_probability')
                        }
                    }
                else:
                    response = {
                        'valid': False,
                        'message': 'Report not found or invalid'
                    }

                self.send_response(200)
                self.send_header('Content-Type', 'application/json')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(json.dumps(response).encode())

            except Exception as e:
                self.send_response(500)
                self.send_header('Content-Type', 'application/json')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(json.dumps({'error': str(e)}).encode())
            return

        if self.path == '/chat':
            try:
                data = json.loads(body)
                user_message = data.get('message', '')
                building_context = data.get('building_context', data.get('buildingData', None))
                collapse_prob = data.get('collapse_prob', data.get('collapseProbability', None))
                history = data.get('history', [])


                if building_context is None:
                    building_context = LAST_BUILDING_DATA if LAST_BUILDING_DATA else "No building data provided."
                if collapse_prob is None:
                    collapse_prob = LAST_COLLAPSE_PROB if LAST_COLLAPSE_PROB is not None else "N/A"

                if isinstance(building_context, dict):
                    building_context = "\n".join([f"{k}: {v}" for k, v in building_context.items()])

                system_msg = ANKA_SYSTEM_PROMPT.format(
                    building_context=building_context,
                    collapse_prob=collapse_prob
                )

                messages = [{"role": "system", "content": system_msg}]
                for h in history[-10:]:
                    messages.append({"role": h.get("role", "user"), "content": h.get("content", "")})
                messages.append({"role": "user", "content": user_message})

                chat_response = llm_client.chat.completions.create(
                    model="claude-sonnet-4-6",
                    messages=messages,
                    max_tokens=8000,
                    temperature=0.7
                )
                reply = chat_response.choices[0].message.content.strip()

                self.send_response(200)
                self.send_header('Content-Type', 'application/json')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(json.dumps({'reply': reply}).encode())

            except Exception as e:
                self.send_response(500)
                self.send_header('Content-Type', 'application/json')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(json.dumps({'error': str(e)}).encode())
            return

        if self.path == '/stt':
            try:
                content_type = self.headers.get('Content-Type', '')
                if 'boundary=' not in content_type:
                    raise ValueError("Invalid Content-Type for multipart data")

                boundary = content_type.split('boundary=')[1].encode()
                parts = body.split(b'--' + boundary)

                audio_data = None
                for part in parts:
                    if b'name="audio"' in part:
                        header_end = part.find(b'\r\n\r\n')
                        if header_end != -1:
                            audio_data = part[header_end + 4:]
                            if audio_data.endswith(b'\r\n'):
                                audio_data = audio_data[:-2]
                        break

                if audio_data:
                    audio_file = io.BytesIO(audio_data)
                    audio_file.name = "audio.webm"

                    response = stt_client.audio.transcriptions.create(
                        model="whisper-1",
                        file=audio_file
                    )

                    self.send_response(200)
                    self.send_header('Content-Type', 'application/json')
                    self.send_header('Access-Control-Allow-Origin', '*')
                    self.end_headers()
                    self.wfile.write(json.dumps({
                        "text": response.text
                    }).encode('utf-8'))
                else:
                    self.send_response(400)
                    self.send_header('Content-Type', 'application/json')
                    self.send_header('Access-Control-Allow-Origin', '*')
                    self.end_headers()
                    self.wfile.write(json.dumps({
                        "error": "No audio data received"
                    }).encode('utf-8'))

            except Exception as e:
                self.send_response(500)
                self.send_header('Content-Type', 'application/json')
                self.send_header('Access-Control-Allow-Origin', '*')
                self.end_headers()
                self.wfile.write(json.dumps({'error': str(e)}).encode())
            return

        self.send_response(404)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.end_headers()
        self.wfile.write(b'Endpoint not found')



_ensure_reports_db()

PORT = 5070
print(f"🚀 Anka server running on http://localhost:{PORT}")
server = HTTPServer(('0.0.0.0', PORT), RequestHandler)
server.serve_forever()